# 06 · Final checks, dataset list and AI note

**Field Shift** (NASA Space Apps 2026), data notebook 6 of 6. *Code written with AI assistance (Claude, Anthropic); run and checked by the team.*

No Earth Engine needed. It reads the data files that notebooks 01–05 left in your Drive folder, then:
1. checks them against each other: every square matches the grid, every record exists, sizes, and the demo field's sections all describe the same outline;
2. merges the demo field's parts (`field_greenness.json`, `field_weather.json`, `field_water.json`, `field_future_soil.json`) into one `field.json`;
3. adds an AI-assistance note and a timestamp to each file's metadata (the Space Apps rules ask for AI-made code and data to be acknowledged in text **and** metadata);
4. writes `data/index.json`, a list of the data files for the app, and `DATA_SOURCES.md`, every dataset with its licence and citation for the README and project page;
5. makes three charts for the pitch: the demo field in the 2024 drought against the wet 2026 season, the ground check below, and hot days at flowering today and by 2041–2060;
6. checks the field's satellite rain and temperature against the weather station of the vetch trial at that same field (Denora et al. 2025).

**How to run (under a minute):** **Runtime → Run all** and pick all the data files in the upload box. It downloads two zip files. Extract `field_shift_for_repo.zip` (right-click → Extract All), then upload the files in its `data` folder to the repo's `data` folder (replacing the old ones), and `DATA_SOURCES.md` to the top folder of the repo. The names inside the zip are always right, even if your browser adds "(1)" to the zip's own name. `field_shift_charts.zip` has the charts for the slides. You can run it as often as you like, for example after 02 and 03 to get a merged `field.json` for the app.

In [ ]:
DRIVE_FOLDER = "field-shift"            # folder for the files (in Google Drive if USE_DRIVE is True)
USE_DRIVE = False                       # False: you pick all the data files with an upload button
AI_NOTE = ("Produced by notebooks written with AI assistance (Claude, Anthropic) and run by the Field Shift team in Google Colab "
           "with Google Earth Engine. Data values come from the cited datasets and were not written by the AI, except a few published "
           "values from the trial at the demo field (its station's rain and temperature, and its soil description), which were copied "
           "by hand from Denora et al. (2025) and are labelled as such.")

In [ ]:
import json, sys, hashlib, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
BASE = Path(DRIVE_FOLDER)
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as e:
        raise RuntimeError(f"Google Drive didn't connect ({str(e)[:80]}). Run this cell again and tick 'Select all' "
                           "(the Google Drive box) in the Google window before pressing Continue.") from None
    BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
elif IN_COLAB:
    BASE = Path("/content") / DRIVE_FOLDER
DATA = BASE / "data"
DATA.mkdir(parents=True, exist_ok=True)
if IN_COLAB and not USE_DRIVE:
    from google.colab import files
    import re
    print("Choose ALL the data files in the upload box below, at once: grid.geojson, area.geojson, field.json (from 01), "
          "field_greenness.json, field_weather.json, field_water.json, field_future_soil.json and every cells_*.json")
    for fname, content in files.upload().items():
        (DATA / re.sub(r" \(\d+\)(?=\.)", "", Path(fname).name)).write_bytes(content)
RUN_AT = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%MZ")

CHECKS = []
def check(ok, text, warn=False, fail=None):
    mark = "✅" if ok else ("⚠️" if warn else "❌")
    line = f"{mark} {text if ok or not fail else fail}"
    CHECKS.append(line)
    print(line)

FILES = ["grid.geojson", "area.geojson", "field.json", "cells_greenness.json", "cells_weather.json",
         "cells_water.json", "cells_future.json", "cells_soil.json"]
PARTS = {"field_greenness.json": ["greenness"], "field_weather.json": ["weather"], "field_water.json": ["water"],
         "field_future_soil.json": ["future", "soil"]}
docs = {}
for name in FILES + list(PARTS):
    path = DATA / name
    if path.exists():
        docs[name] = json.loads(path.read_text())
    check(path.exists(), f"{name} found", warn=name not in ("grid.geojson", "field.json"),
          fail=f"{name} missing: run the notebook that makes it")

# merge each notebook's part of the demo field into field.json
field = docs["field.json"]
for name, secs in PARTS.items():
    part = docs.pop(name, None)
    if not part:
        continue
    for sec in secs:
        if sec in part:
            field[sec] = part[sec]
            field[sec].setdefault("field_key", part["meta"].get("field_key"))
            field["meta"].setdefault("made_by", {})[sec] = part["meta"].get("made_by")
    field["meta"].setdefault("sources", {}).update(part["meta"].get("sources", {}))
field["meta"]["updated"] = RUN_AT
print("field.json now has:", [k for k in field if k != "meta"])

## 1. Do the files fit together?

In [ ]:
grid = docs["grid.geojson"]
ids = {f["id"] for f in grid["features"]}
farm = {f["id"] for f in grid["features"] if f["properties"].get("farm")}
check(len(ids) == len(grid["features"]), f"Grid: {len(ids):,} unique squares, {len(farm):,} farmland")

for name in [n for n in FILES if n.startswith("cells_") and n in docs]:
    d = docs[name]
    keys = set(d["cells"])
    stray = keys - ids
    cover = len(keys & farm) / max(len(farm), 1)
    check(not stray, f"{name}: every key is a grid square", fail=f"{name}: {len(stray)} keys are not in grid.geojson (rerun it after 01)")
    check(cover >= 0.95, f"{name}: covers {len(keys & farm):,} of {len(farm):,} farmland squares ({cover:.1%})", warn=True)
    if "records" in d:
        refs = {v["rec"] if isinstance(v, dict) else v for v in d["cells"].values()}
        missing = refs - set(d["records"])
        check(not missing, f"{name}: all {len(refs)} shared records exist", fail=f"{name}: {len(missing)} records missing")
    srcs = set(d["meta"].get("sources", {}))
    used = {s.strip() for ind in d["meta"].get("indicators", {}).values() for s in str(ind.get("source", "")).split(",") if s.strip()}
    check(used <= srcs, f"{name}: every indicator's source is listed", warn=True,
          fail=f"{name}: sources not listed in meta: {sorted(used - srcs)}")

fsq = field["field"]["square"]
check(fsq in ids, f"Demo field square {fsq} is in the grid")
made = field["meta"].get("made_by", {})
outline_key = hashlib.md5(json.dumps(field["field"]["outline"], sort_keys=True).encode()).hexdigest()[:8]
outline_time = made.get("field", "").split("(")[-1].rstrip(")")
for part in ("greenness", "weather", "water", "future", "soil"):
    if part not in field:
        check(False, f"field.json has '{part}'", warn=True, fail=f"field.json has no '{part}' section yet: run its notebook")
        continue
    key = field[part].get("field_key")
    if key is not None:
        ok = key == outline_key
    else:                                   # older runs: fall back on times
        ok = made.get(part, "").split("(")[-1].rstrip(")") >= outline_time
    check(ok, f"field.json '{part}' matches the current field outline", warn=True,
          fail=f"field.json '{part}' was worked out for an earlier field outline: rerun that notebook")

## 2. The demo field, drought against wet year
The same two seasons seen by four independent measurements. If they all agree, that's strong evidence the data is right, and a strong story for the pitch.

In [ ]:
def season_value(section, key, season):
    for r in field.get(section, {}).get("seasons", []):
        if r.get("season") == season:
            return r.get(key)
    return None

def normal(section, key):
    t = field.get(section, {}).get("typical", {}).get(key)
    return t[1] if t else None

ROWS = [("Rain October-May (mm)", "weather", "rain_season", "rain_season"),
        ("Root-zone soil water in spring (%)", "water", "sm_spring", "sm_spring"),
        ("Peak greenness (NDVI)", "greenness", "peak", "peak"),
        ("Water used by the crop, Nov-Jun (mm)", "water", "water_use_modis", "water_use_modis")]
table = []
for label, section, key, tkey in ROWS:
    table.append({"measure": label, "normal": normal(section, tkey),
                  "2024 (drought)": season_value(section, key, 2024), "2026 (wet)": season_value(section, key, 2026)})
contrast = pd.DataFrame(table).set_index("measure")
print(contrast.to_string())
agree = [(r["2024 (drought)"] < r["normal"]) for _, r in contrast.iterrows()
         if r["normal"] is not None and r["2024 (drought)"] is not None]
check(len(agree) >= 3 and all(agree), f"All {len(agree)} measures put 2024 below normal at the demo field", warn=True,
      fail=f"Only {sum(agree)} of {len(agree)} measures put 2024 below normal: look at the table above")

In [ ]:
import matplotlib.pyplot as plt
plot = contrast.dropna(how="all")
fig, axes = plt.subplots(1, len(plot), figsize=(3.2 * len(plot), 3.6))
axes = np.atleast_1d(axes)
for ax, (label, r) in zip(axes, plot.iterrows()):
    vals = [r["normal"], r["2024 (drought)"], r["2026 (wet)"]]
    ax.bar(["Normal", "2024", "2026"], [np.nan if v is None else v for v in vals], color=["#7f8c8d", "#c0392b", "#1f6fb2"])
    ax.set_title(label, fontsize=9, loc="left")
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("Demo field, Genzano di Lucania: from drought (season 2024) to a wet year (season 2026)", x=0.01, ha="left")
plt.tight_layout()
plt.savefig(BASE / "drought_to_wet_field.png", dpi=150)
plt.show()

## 2b. Ground check: the weather station at the demo field
The vetch trial at this field had its own weather station (Denora et al. 2025, *Italian Journal of Agronomy* 20, 100045). The paper gives each season's total rain and mean temperature but not the exact months it counted, so the satellite rain is added up over three windows: October–May (the app's season rain), November–June, and sowing to harvest (part months counted by their share of days). The station numbers are typed in from the paper; everything else comes from `field_weather.json`. If anything goes wrong here, the notebook carries on.

In [ ]:
# Typed in by hand from Denora et al. (2025): sowing and harvest dates from the methods, rain and mean temperature from Section 3.1.
STATION = {
    "cite": ("Denora M. et al. (2025). Durum wheat-vetch intercropping in Mediterranean agroecological systems: effects on yield, "
             "nitrogen balance, and farm profitability. Italian Journal of Agronomy 20, 100045. https://doi.org/10.1016/j.ijagro.2025.100045"),
    "where": "weather station at the trial site, La Generale cooperative, Genzano di Lucania (40.82460 N, 16.09348 E)",
    "seasons": {2023: {"sown": "2022-12-22", "harvested": "2023-07-13", "rain_mm": 673.6, "t_mean_c": 13.58},
                2024: {"sown": "2023-12-09", "harvested": "2024-06-24", "rain_mm": 314.6, "t_mean_c": 15.47}}}
MAIN = {"chirps": "CHIRPS", "imerg": "IMERG", "era5": "ERA5-Land"}   # the three rain sources whose median the app uses

def month_series(mon, key):
    vals = pd.to_numeric(pd.Series(mon.get(key, [None] * len(mon["months"]))), errors="coerce").values
    return pd.Series(vals, index=pd.PeriodIndex(mon["months"], freq="M"))

def share_of_months(start, end):
    """Share of each calendar month that falls between two dates (both included)."""
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    out = {}
    for p in pd.period_range(start, end, freq="M"):
        first, last = max(start, p.start_time.normalize()), min(end, p.end_time.normalize())
        out[p] = ((last - first).days + 1) / p.days_in_month
    return pd.Series(out)

def window_total(series, share):
    vals = series.reindex(share.index)
    return None if vals.isna().any() else float((vals * share).sum())

def window_mean(series, share):
    vals = series.reindex(share.index)
    days = share * np.array([p.days_in_month for p in share.index])
    return None if vals.isna().any() else float((vals * days).sum() / days.sum())

ground, rain_rows, temp_rows = None, [], []
try:
    mon = field["weather"]["monthly"]
    rain = {s: month_series(mon, f"rain_{s}") for s in MAIN}
    t_mean = (month_series(mon, "tmax_era5") + month_series(mon, "tmin_era5")) / 2
    for season, st in STATION["seasons"].items():
        label = f"{season - 1}-{str(season)[2:]}"
        windows = {"Oct-May": share_of_months(f"{season - 1}-10-01", f"{season}-05-31"),
                   "Nov-Jun": share_of_months(f"{season - 1}-11-01", f"{season}-06-30"),
                   "sowing to harvest": share_of_months(st["sown"], st["harvested"])}
        for wname, share in windows.items():
            tot = {s: window_total(rain[s], share) for s in MAIN}
            ok = [v for v in tot.values() if v is not None]
            med = float(np.median(ok)) if ok else None
            row = {"season": label, "window": wname, "station_mm": st["rain_mm"], "satellites_mm": None if med is None else round(med),
                   "difference_pct": None if med is None else round(100 * (med - st["rain_mm"]) / st["rain_mm"])}
            row.update({f"{s}_mm": None if v is None else round(v) for s, v in tot.items()})
            rain_rows.append(row)
        t_sat = window_mean(t_mean, windows["sowing to harvest"])
        row_t = {"season": label, "window": "sowing to harvest", "station_c": st["t_mean_c"],
                 "era5_land_c": None if t_sat is None else round(t_sat, 1),
                 "difference_c": None if t_sat is None else round(t_sat - st["t_mean_c"], 1)}
        fix = field["weather"].get("station_corrected")
        if fix and t_sat is not None:                 # ERA5-Land after notebook 03's correction with a nearby station
            add = pd.Series([(fix["offset_c"]["tmax"][p.month - 1] + fix["offset_c"]["tmin"][p.month - 1]) / 2
                             for p in t_mean.index], index=t_mean.index)
            t_fix = window_mean(t_mean + add, windows["sowing to harvest"])
            row_t.update({"era5_land_corrected_c": round(t_fix, 1), "difference_corrected_c": round(t_fix - st["t_mean_c"], 1)})
        temp_rows.append(row_t)

    show = pd.DataFrame(rain_rows).rename(columns={"station_mm": "station", "satellites_mm": "satellites (median)",
                                                   "difference_pct": "difference %", **{f"{s}_mm": n for s, n in MAIN.items()}})
    print("Rain (mm): the trial's station against the satellite sources at the field\n" + show.to_string(index=False))
    print("\nMean temperature (°C), sowing to harvest\n" + pd.DataFrame(temp_rows).to_string(index=False))

    sh = {r["season"]: r for r in rain_rows if r["window"] == "sowing to harvest" and r["satellites_mm"] is not None}
    if len(sh) == 2:
        wet, dry = sh["2022-23"], sh["2023-24"]
        st_drop, sat_drop = 1 - dry["station_mm"] / wet["station_mm"], 1 - dry["satellites_mm"] / wet["satellites_mm"]
        check(sat_drop >= 0.2, f"Station and satellites agree 2023-24 was much drier than 2022-23: station -{st_drop:.0%}, "
              f"satellites -{sat_drop:.0%} (sowing to harvest)", warn=True,
              fail=f"Satellites show 2023-24 only {sat_drop:.0%} drier than 2022-23, the station {st_drop:.0%}")
        worst = max(abs(r["difference_pct"]) for r in sh.values())
        check(worst <= 25, f"Satellite rain within {worst}% of the trial's station in both seasons (sowing to harvest)", warn=True,
              fail=f"Satellite rain differs from the trial's station by up to {worst}% (sowing to harvest): say so on the evidence screen")
    td = [abs(r["difference_c"]) for r in temp_rows if r["difference_c"] is not None]
    if td:
        check(max(td) <= 1.5, f"ERA5-Land mean temperature within {max(td):.1f} °C of the trial's station", warn=True,
              fail=f"ERA5-Land mean temperature differs from the trial's station by up to {max(td):.1f} °C")

    ground = {"what": "Satellite rain (median of CHIRPS, IMERG, ERA5-Land) and ERA5-Land temperature at the demo field, "
                      "checked against the weather station of the vetch trial at the same field",
              "station": {"cite": STATION["cite"], "where": STATION["where"],
                          "seasons": {str(k): v for k, v in STATION["seasons"].items()},
                          "note": "station values copied by hand from the paper; used only as a check"},
              "rain": rain_rows, "temperature": temp_rows,
              "windows": "the paper does not say which months its season totals cover, so three windows are shown; "
                         "part months are counted by their share of days"}
    field["weather"]["ground_check"] = ground
except Exception as e:
    check(False, "Ground check done", warn=True, fail=f"Ground check skipped ({type(e).__name__}: {str(e)[:100]})")

In [ ]:
if ground:
    import matplotlib.pyplot as plt
    rows = [r for r in rain_rows if r["window"] == "sowing to harvest"]
    bars = [("Station at the field", "station_mm", "#2c3e50"), ("Satellites (median)", "satellites_mm", "#e67e22"),
            ("CHIRPS", "chirps_mm", "#f5cba7"), ("IMERG", "imerg_mm", "#f8d9b6"), ("ERA5-Land", "era5_mm", "#fbe5cf")]
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    width = 0.16
    for i, (name, key, colour) in enumerate(bars):
        vals = [np.nan if r[key] is None else r[key] for r in rows]
        ax.bar(np.arange(len(rows)) + (i - 2) * width, vals, width, label=name, color=colour, edgecolor="#555", linewidth=0.4)
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels([f"{r['season']} season" for r in rows])
    ax.set_ylabel("Rain, sowing to harvest (mm)")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8, ncol=2, frameon=False)
    ax.set_title("Demo field: satellite rain against the vetch trial's own weather station", loc="left", fontsize=10)
    fig.text(0.01, 0.01, "Station: Denora et al. 2025, Italian Journal of Agronomy 20, 100045", fontsize=7, color="#555")
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    plt.savefig(BASE / "ground_check_field.png", dpi=150)
    plt.show()

## 2c. Heat at flowering, today and by 2041–2060
Days at 30 °C or more while durum flowers and fills its grain (15 April – 15 June), at the demo field. The bars are a typical season. Only the 2041–2060 bars get lines, showing how far the climate models disagree. How much today's count changes from one season to the next is a different kind of range, so it is written under the chart instead of drawn.

In [ ]:
fut = field.get("future", {})
now_hot = (fut.get("now") or {}).get("hot")
proj = {s: fut.get("projected", {}).get(s, {}).get("hot") for s in ("ssp245", "ssp585")}
heat = None
if now_hot and all(proj.values()):
    import matplotlib.pyplot as plt
    past = str(fut.get("periods", {}).get("past", "2001-2020")).split(" ")[0]
    later = str(fut.get("periods", {}).get("future", "2041-2060"))
    labels = [f"Today\n({past})", f"{later}\nmiddle (SSP2-4.5)", f"{later}\nhigh (SSP5-8.5)"]
    mids = [now_hot[1], proj["ssp245"][1], proj["ssp585"][1]]
    lows = [proj["ssp245"][0], proj["ssp585"][0]]
    highs = [proj["ssp245"][2], proj["ssp585"][2]]
    colours = ["#77756e", "#ec8f5e", "#c2491a"]          # neutral for today, then one orange that darkens with emissions
    inks = ["#ffffff", "#0b0b0b", "#ffffff"]
    fig, ax = plt.subplots(figsize=(7, 4.9))
    x = np.arange(3)
    ax.bar(x, mids, width=0.5, color=colours)
    ax.errorbar(x[1:], mids[1:], yerr=[np.array(mids[1:]) - lows, np.array(highs) - mids[1:]],
                fmt="none", ecolor="#52514e", elinewidth=1.2, capsize=6)
    for i, (xi, m) in enumerate(zip(x, mids)):
        y = m / 2 if i == 0 else min(m / 2, lows[i - 1] * 0.8)
        ax.text(xi, y, f"{round(m, 1):g}", ha="center", va="center", color=inks[i], fontsize=14, fontweight="bold")
    ax.set_xticks(x, labels)
    ax.set_ylim(0, max(highs + [now_hot[1]]) * 1.15)
    ax.set_ylabel("Hot days, 15 April - 15 June")
    ax.set_title("Demo field: days at 30 °C or more while durum flowers", loc="left")
    ax.grid(axis="y", color="#e1e0d9", linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c3c2b7")
    notes = [f"Bars: a typical season. Lines: how far the {len(fut.get('models', [])) or 'climate'} climate models disagree "
             "(10th to 90th percentile).",
             f"Today the count runs from {now_hot[0]:g} in a cool season to {now_hot[2]:g} in a hot one "
             f"(10th to 90th percentile of {past})."]
    diffs = [r["difference_c"] for r in temp_rows if r.get("difference_c") is not None]
    if diffs and max(diffs) < -0.5:
        notes.append(f"From ERA5-Land, which reads {-max(diffs):.1f}-{-min(diffs):.1f} °C cooler than the trial's own weather "
                     "station, so all these counts are probably on the low side.")
    elif diffs and min(diffs) > 0.5:
        notes.append(f"From ERA5-Land, which reads {min(diffs):.1f}-{max(diffs):.1f} °C warmer than the trial's own weather "
                     "station, so all these counts are probably on the high side.")
    fig.text(0.01, 0.01, "\n".join(notes), fontsize=7.5, color="#52514e", va="bottom")
    plt.tight_layout(rect=(0, 0.04 + 0.03 * len(notes), 1, 1))
    plt.savefig(BASE / "future_heat_field.png", dpi=150)
    plt.show()
    heat = {"today": now_hot, "ssp245": proj["ssp245"], "ssp585": proj["ssp585"]}
else:
    print("No future section in field.json yet (run 05 first): heat chart skipped")

## 3. AI note, index of files and dataset list

In [ ]:
if field.get("weather", {}).get("station_corrected") and "annals" not in AI_NOTE:
    AI_NOTE += (" The monthly station temperatures used to correct the demo field were read from the Basilicata hydrological "
                "annals with code written with AI assistance; they are not in this repository.")
for name, d in docs.items():
    d.setdefault("meta", {})
    d["meta"]["ai_assistance"] = AI_NOTE
    d["meta"]["checked"] = f"notebooks/06_check_and_sources.ipynb ({RUN_AT})"
    (DATA / name).write_text(json.dumps(d, ensure_ascii=False, separators=(",", ":"), allow_nan=False))

index = {"project": "Field Shift (NASA Space Apps Challenge 2026)", "updated": RUN_AT, "ai_assistance": AI_NOTE,
         "conventions": {"low_typical_high": "10th percentile, median, 90th percentile",
                         "season": "named by harvest year: season 2024 = September 2023 to August 2024",
                         "days": "day numbers count from 1 September", "grid": "EEA 1 km reference grid, EPSG:3035"},
         "files": {name: {"bytes": (DATA / name).stat().st_size,
                          "made_by": d["meta"].get("made_by"),
                          "what": {"grid.geojson": "1 km squares with land cover, durum share, height, slope",
                                   "area.geojson": "basin outline", "field.json": "the demo field in detail",
                                   "cells_greenness.json": "greenness per farmland square",
                                   "cells_weather.json": "rain and temperature per square (records[cells[id]])",
                                   "cells_water.json": "water need, water use, soil moisture (records[cells[id].rec] + cells[id])",
                                   "cells_future.json": "change by 2041-2060 (records[cells[id]])",
                                   "cells_soil.json": "soil per square"}.get(name, "")}
                   for name, d in docs.items()}}
(DATA / "index.json").write_text(json.dumps(index, ensure_ascii=False, indent=1))
sizes = {name: (DATA / name).stat().st_size / 1e6 for name in docs}           # measured after writing the merged files
check(all(v < 5 for v in sizes.values()), f"All files under 5 MB; total {sum(sizes.values()):.1f} MB")

sources = {}
for d in docs.values():
    for k, v in d.get("meta", {}).get("sources", {}).items():
        sources.setdefault(k, v)
for k, v in field.get("meta", {}).get("sources", {}).items():
    sources.setdefault(k, v)
USED_FOR = {"hydrosheds": "basin outline", "worldcover": "cropland mask", "corine": "non-irrigated arable mask",
            "eucropmap": "durum wheat share, demo-field check", "nasadem": "height and slope",
            "modis_ndvi": "greenness calendar and season dates", "hls": "30 m greenness at the demo field",
            "sentinel2": "10 m greenness and red-edge at the demo field", "era5_land": "temperature, rain, ET0, soil water (check)",
            "chirps": "rain", "imerg": "rain", "gsmap": "rain at the demo field (comparison)", "power": "checks at the demo field",
            "modis_et": "water used by crops", "smap": "soil moisture", "terraclimate": "ET0 check", "fao56": "ET0 formula, crop factors",
            "nex_gddp_cmip6": "climate change by 2041-2060", "soilgrids": "soil properties", "soilgrids_water": "water-holding capacity",
            "basilicata_stations": "temperature correction at the demo field (station records not in this repository)"}
lines = ["# Data sources", "",
         f"All data in `data/` comes from the datasets below, processed by the notebooks in `ColabNotebooks/` (last checked {RUN_AT}).",
         ("Every dataset in the table is free and openly licensed, except the Basilicata station records (see their row)."
          if field.get("weather", {}).get("station_corrected") else "Every dataset is free and openly licensed.")
         + (" Rain and temperature published for the trial at the demo field (Denora et al. 2025) are quoted, with the citation, "
            "only to check results: see Ground check below and `weather.ground_check` in `field.json`." if ground else "")
         + (" Basilicata weather-station records were used only to correct the demo field's temperatures; they are not in this repository."
            if field.get("weather", {}).get("station_corrected") else ""), "",
         "| Dataset | Used for | Earth Engine ID or service | Resolution | Licence | Citation |", "|---|---|---|---|---|---|"]
for k, v in sorted(sources.items(), key=lambda kv: kv[1].get("name", kv[0])):
    res = f"{v['res_m']:,} m" if isinstance(v.get("res_m"), (int, float)) else "-"
    lines.append(f"| {v.get('name', k)} | {USED_FOR.get(k, '')} | `{v.get('ee_id', '-')}` | {res} | {v.get('licence', '')} | {v.get('cite', '')} |")
lines += ["", "The 1 km grid follows the EEA reference grid (ETRS89-LAEA, EPSG:3035).", ""]
if ground:
    lines += ["## Ground check", "",
              "Rain and temperature at the demo field were checked against the weather station of the vetch trial at that field. "
              "The paper gives season totals without saying which months they cover, so three windows are shown. "
              "The station values are copied by hand from the paper and labelled as such in `field.json`. "
              "They are used only for this check: no data was changed to match them.", "",
              "| Season | Months | Station rain (mm) | Satellite rain, median of CHIRPS, IMERG, ERA5-Land (mm) | Difference |",
              "|---|---|---|---|---|"]
    for r in rain_rows:
        diff = "-" if r["difference_pct"] is None else f"{r['difference_pct']:+d}%"
        lines.append(f"| {r['season']} | {r['window']} | {r['station_mm']} | {r['satellites_mm']} | {diff} |")
    fixed = any("era5_land_corrected_c" in r for r in temp_rows)
    lines += ["", "| Season | Station mean temperature (°C) | ERA5-Land, sowing to harvest (°C) | Difference (°C) |"
              + (" ERA5-Land corrected with a nearby station (°C) | Difference (°C) |" if fixed else ""),
              "|---|---|---|---|" + ("---|---|" if fixed else "")]
    for r in temp_rows:
        lines.append(f"| {r['season']} | {r['station_c']} | {r['era5_land_c']} | {r['difference_c']} |"
                     + (f" {r.get('era5_land_corrected_c', '-')} | {r.get('difference_corrected_c', '-')} |" if fixed else ""))
    lines += ["", f"Station: {STATION['cite']}", ""]
lines += ["## AI assistance", "", AI_NOTE, ""]
(BASE / "DATA_SOURCES.md").write_text("\n".join(lines))
check(len(sources) >= 10, f"DATA_SOURCES.md lists {len(sources)} datasets")
print("\n".join(lines[:12]))

## 4. Downloads and what to paste back

In [ ]:
import zipfile
REPO_ZIP, CHART_ZIP = BASE / "field_shift_for_repo.zip", BASE / "field_shift_charts.zip"
with zipfile.ZipFile(REPO_ZIP, "w", zipfile.ZIP_DEFLATED) as z:       # exact file names, whatever the browser calls the zip
    for name in list(docs) + ["index.json"]:
        z.write(DATA / name, f"data/{name}")
    z.write(BASE / "DATA_SOURCES.md", "DATA_SOURCES.md")
charts = [BASE / n for n in ("drought_to_wet_field.png", "ground_check_field.png", "future_heat_field.png") if (BASE / n).exists()]
with zipfile.ZipFile(CHART_ZIP, "w") as z:
    for p in charts:
        z.write(p, p.name)
print("\n".join(CHECKS))
if IN_COLAB:
    from google.colab import files
    files.download(str(REPO_ZIP))
    files.download(str(CHART_ZIP))
    print("Extract field_shift_for_repo.zip, then upload the files in its data folder to the repo's data folder (replacing the old ones) "
          "and DATA_SOURCES.md to the top folder. field_shift_charts.zip has the charts for the slides.")
print("=== Paste this back to Claude ===")
print(f"06 run {RUN_AT}; files {sorted(docs)}; sizes MB {({k: round(v, 2) for k, v in sizes.items()})}")
print(contrast.to_string())
if ground:
    print(pd.DataFrame(rain_rows).to_string(index=False))
    print(pd.DataFrame(temp_rows).to_string(index=False))
print("heat at flowering:", json.dumps(heat), "| charts:", [p.name for p in charts])
print("\n".join(CHECKS))